### Baseline Evaluation on CityScape data


In [ ]:
import os
import glob

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
from dotenv import load_dotenv
import kagglehub
import sys
import subprocess
import gdown

# ── 1. Download dataset ─────────────────────────────────────────────────────

load_dotenv()
from google.colab import userdata
api_key = userdata.get('KAGGLE_API_KEY')

# api_key = os.environ.get("KAGGLE_API_KEY") or os.environ.get("KAGGLE_KEY")
if api_key:
    os.environ["KAGGLE_KEY"] = api_key
    # KGAT tokens don't need a username; set a placeholder if missing
    if not os.environ.get("KAGGLE_USERNAME"):
        os.environ["KAGGLE_USERNAME"] = "__token__"

print("Downloading Cityscapes dataset …")
dataset_path = kagglehub.dataset_download("shuvoalok/cityscapes")
print(f"Dataset path: {dataset_path}")

# Locate val images and segmentation maps
# Original path: os.path.join(dataset_path, "cityscapes", "val", "image")
val_img_dir = os.path.join(dataset_path, "val", "img")
val_seg_dir = os.path.join(dataset_path, "val", "label")

# Fallback: search recursively if the expected structure differs
if not os.path.isdir(val_img_dir):
    for root, dirs, _ in os.walk(dataset_path):
        if "val" in dirs:
            candidate = os.path.join(root, "val")
            sub = os.listdir(candidate)
            if "img" in sub: # Changed 'image' to 'img'
                val_img_dir = os.path.join(candidate, "img") # Changed 'image' to 'img'
                val_seg_dir = os.path.join(candidate, "label")
                break
    print(f"Resolved val dirs → images: {val_img_dir}, labels: {val_seg_dir}")

assert os.path.isdir(val_img_dir), f"Image dir not found: {val_img_dir}"
assert os.path.isdir(val_seg_dir), f"Label dir not found: {val_seg_dir}"

ROAD_COLOR = (128, 64, 128)
COLOR_TOLERANCE = 5  # max per-channel difference to handle compression artifacts

# Debug: inspect label format and colors
sample_labels = sorted(glob.glob(os.path.join(val_seg_dir, "*.*")))[:3]
print(f"\nLabel file samples: {[os.path.basename(f) for f in sample_labels]}")
for lbl_path in sample_labels:
    lbl = Image.open(lbl_path).convert("RGB")
    lbl_np = np.array(lbl)
    # Find unique colors (sample to avoid huge output)
    pixels = lbl_np.reshape(-1, 3)
    unique_colors = np.unique(pixels, axis=0)
    print(f"  {os.path.basename(lbl_path)}: size={lbl.size}, format={lbl.format}, "
          f"unique colors={len(unique_colors)}")
    # Check if road color exists
    road_match = np.all(pixels == ROAD_COLOR, axis=1).sum()
    print(f"    Road color (128,64,128) pixels: {road_match}/{len(pixels)}")
    # Show top 5 most common colors
    colors, counts = np.unique(pixels, axis=0, return_counts=True)
    top5 = np.argsort(-counts)[:5]
    for i in top5:
        print(f"    Color {tuple(colors[i])}: {counts[i]} pixels ({100*counts[i]/len(pixels):.1f}%)")
print()

# ── 2. Dataset class ────────────────────────────────────────────────────────

IMG_SIZE = (512, 1024)  # H, W

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

img_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class CityscapesRoad(Dataset):
    def __init__(self, img_dir, seg_dir):
        # Changed glob pattern from "*.* развития" to "*.*"
        self.images = sorted(glob.glob(os.path.join(img_dir, "*.*")))
        self.labels = sorted(glob.glob(os.path.join(seg_dir, "*.*")))
        assert len(self.images) == len(self.labels), (
            f"Mismatch: {len(self.images)} images vs {len(self.labels)} labels"
        )
        print(f"Found {len(self.images)} val image/label pairs")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        seg = Image.open(self.labels[idx]).convert("RGB")

        # Resize
        img = img_transform(img)
        seg = seg.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.NEAREST)

        # Binary road mask from RGB color (tolerance for compression artifacts)
        seg_np = np.array(seg).astype(np.int16)  # (H, W, 3)
        diff = np.abs(seg_np - np.array(ROAD_COLOR, dtype=np.int16))
        road_mask = (diff.max(axis=2) <= COLOR_TOLERANCE).astype(np.float32)

        return img, torch.from_numpy(road_mask)


val_dataset = CityscapesRoad(val_img_dir, val_seg_dir)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

# ── 3. Model: DeepLabV3+ ResNet101 pretrained on Cityscapes ─────────────────

REPO_DIR = "DeepLabV3Plus-Pytorch"
CHECKPOINT = "best_deeplabv3plus_resnet101_cityscapes_os16.pth"
GDRIVE_ID = "1t7TC8mxQaFECt4jutdq_NMnWxdm6B-Nb"

# Clone model repo if needed
if not os.path.isdir(REPO_DIR):
    print("Cloning DeepLabV3Plus-Pytorch …")
    subprocess.run(["git", "clone", "https://github.com/VainF/DeepLabV3Plus-Pytorch.git"], check=True)

# Download Cityscapes checkpoint if needed
if not os.path.isfile(CHECKPOINT):
    print("Downloading Cityscapes-pretrained checkpoint …")
    gdown.download(id=GDRIVE_ID, output=CHECKPOINT, quiet=False)

sys.path.insert(0, REPO_DIR)
from network.modeling import deeplabv3plus_resnet101

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = deeplabv3plus_resnet101(num_classes=19, output_stride=16)
checkpoint = torch.load(CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model = model.to(device).eval()

# Cityscapes trainId: road = 0
NUM_CLASSES = 19
ROAD_CLASS = 0

# ── 4. Inference on val set ─────────────────────────────────────────────────

print("Running inference …")

all_preds_list = []
all_probs_list = []
all_gt_list = []

with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc="Inference"):
        imgs = imgs.to(device)
        output = model(imgs)  # (B, 19, H, W)
        probs = torch.softmax(output, dim=1)
        road_prob = probs[:, ROAD_CLASS, :, :].cpu().numpy()
        road_pred = (output.argmax(dim=1) == ROAD_CLASS).cpu().numpy()
        all_preds_list.append(road_pred.astype(np.float32))
        all_probs_list.append(road_prob.astype(np.float32))
        all_gt_list.append(masks.numpy())

all_preds = np.concatenate(all_preds_list, axis=0)
all_probs = np.concatenate(all_probs_list, axis=0)
all_gt = np.concatenate(all_gt_list, axis=0)

N = all_preds.shape[0]
print(f"Processed {N} images")

# ── 4b. Save segmentation visualizations ────────────────────────────────────

OUTPUT_DIR = "segmentation_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
NUM_SAVE = min(20, N)  # save first 20 images

print(f"Saving {NUM_SAVE} visualization images to {OUTPUT_DIR}/ …")
for i in range(NUM_SAVE):
    # Load original image resized to match mask dimensions
    orig = Image.open(val_dataset.images[i]).convert("RGB")
    orig = orig.resize((IMG_SIZE[1], IMG_SIZE[0]))
    orig_np = np.array(orig)

    pred_mask = all_preds[i].astype(bool)
    gt_mask = all_gt[i].astype(bool)

    # Create overlay: green = prediction, red = GT, yellow = overlap
    overlay = orig_np.copy().astype(np.float32)
    # GT in red
    overlay[gt_mask, 0] = np.clip(overlay[gt_mask, 0] + 100, 0, 255)
    overlay[gt_mask, 1] = overlay[gt_mask, 1] * 0.5
    overlay[gt_mask, 2] = overlay[gt_mask, 2] * 0.5
    # Prediction in green
    overlay[pred_mask, 1] = np.clip(overlay[pred_mask, 1] + 100, 0, 255)
    overlay[pred_mask, 0] = overlay[pred_mask, 0] * 0.5
    overlay[pred_mask, 2] = overlay[pred_mask, 2] * 0.5
    # Overlap (both GT and pred) in yellow
    overlap = gt_mask & pred_mask
    overlay[overlap, 0] = np.clip(orig_np[overlap, 0].astype(np.float32) * 0.5 + 127, 0, 255)
    overlay[overlap, 1] = np.clip(orig_np[overlap, 1].astype(np.float32) * 0.5 + 127, 0, 255)
    overlay[overlap, 2] = orig_np[overlap, 2] // 2

    # Side-by-side: original | GT mask | prediction mask | overlay
    gt_vis = np.stack([gt_mask * 255] * 3, axis=-1).astype(np.uint8)
    pred_vis = np.stack([pred_mask * 255] * 3, axis=-1).astype(np.uint8)
    combined = np.concatenate([orig_np, gt_vis, pred_vis, overlay.astype(np.uint8)], axis=1)

    fname = os.path.basename(val_dataset.images[i])
    Image.fromarray(combined).save(os.path.join(OUTPUT_DIR, f"seg_{fname}"))

print(f"Saved to {OUTPUT_DIR}/ (layout: original | GT | prediction | overlay)")
print("Overlay colors: red=GT only, green=pred only, yellow=overlap\n")

# ── 5. Mean IoU (from scratch) ──────────────────────────────────────────────


def compute_mean_iou(preds, gt):
    """Per-image IoU for road class, averaged over all images."""
    ious = []
    for i in range(len(preds)):
        p = preds[i].astype(bool)
        g = gt[i].astype(bool)
        intersection = np.logical_and(p, g).sum()
        union = np.logical_or(p, g).sum()
        if union == 0:
            ious.append(1.0 if intersection == 0 else 0.0)
        else:
            ious.append(intersection / union)
    return float(np.mean(ious))


mean_iou = compute_mean_iou(all_preds, all_gt)

# ── 6. mAP (from scratch) ───────────────────────────────────────────────────


def compute_ap(probs_flat, gt_flat):
    """Average Precision via 101-point interpolated precision-recall curve."""
    thresholds = np.linspace(0, 1, 101)
    precisions = []
    recalls = []

    total_pos = gt_flat.sum()
    if total_pos == 0:
        return 0.0

    for t in thresholds:
        pred = (probs_flat >= t).astype(bool)
        tp = np.logical_and(pred, gt_flat).sum()
        fp = np.logical_and(pred, ~gt_flat).sum()
        fn = np.logical_and(~pred, gt_flat).sum()

        prec = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        precisions.append(prec)
        recalls.append(rec)

    precisions = np.array(precisions)
    # Fixed typo: changed 'recells' to 'recalls'
    recalls = np.array(recalls)

    # Sort by recall ascending
    order = np.argsort(recalls)
    recalls = recalls[order]
    precisions = precisions[order]

    # Interpolate: for each recall level, precision = max precision at >= recall
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])

    # Area under curve (trapezoidal)
    ap = np.trapz(precisions, recalls)
    return float(ap)


# Flatten all predictions/gt across images for AP computation
probs_flat = all_probs.ravel()
gt_flat = all_gt.ravel().astype(bool)

# AP for road class
ap_road = compute_ap(probs_flat, gt_flat)

# AP for background class (invert)
ap_bg = compute_ap(1.0 - probs_flat, ~gt_flat)

mean_ap = (ap_road + ap_bg) / 2.0

# ── 7. Print results ────────────────────────────────────────────────────────

print("\n" + "=" * 50)
print("RESULTS")
print("=" * 50)
print(f"  Road class index (Cityscapes trainId): {ROAD_CLASS}")
print(f"  Mean IoU (road):         {mean_iou:.4f}")
print(f"  AP (road):               {ap_road:.4f}")
print(f"  AP (background):         {ap_bg:.4f}")
print(f"  mAP:                     {mean_ap:.4f}")
print("=" * 50)

### Auto annotate local video

In [ ]:
import os
import glob
import argparse

import cv2
import numpy as np
from PIL import Image
import sys # Import sys to check for ipykernel


def annotate_road(img_bgr):
    """
    Generate a binary road mask from a dashcam image.

    Strategy:
      1. HSV color filter: road is gray (low saturation), medium brightness.
      2. Texture filter: road is smooth (low local gradient variance).
      3. Trapezoidal spatial prior: road fans out from vanishing point.
      4. Combine color + texture + spatial, then flood-fill from bottom-center.
      5. Morphological cleanup.

    Returns:
        Binary mask (H, W) with 255 = road, 0 = not-road.
    """
    h, w = img_bgr.shape[:2]

    # --- Step 1: HSV color filter ---
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    sat = hsv[:, :, 1].astype(np.float32)
    val = hsv[:, :, 2].astype(np.float32)
    hue = hsv[:, :, 0].astype(np.float32)

    # Road asphalt: low saturation, mid-range value
    # Tighter thresholds to exclude walls/buildings
    color_mask = (sat < 50) & (val > 50) & (val < 180)

    # Exclude greenish pixels (vegetation has hue ~35-85 in OpenCV)
    green_mask = (hue > 25) & (hue < 90) & (sat > 20)
    color_mask = color_mask & ~green_mask

    # --- Step 2: Texture filter (smooth regions) ---
    # Road surface is smoother than walls, vegetation, gravel
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    # Local variance in a 15x15 window
    gray_f = gray.astype(np.float32)
    blur = cv2.GaussianBlur(gray_f, (15, 15), 0)
    blur_sq = cv2.GaussianBlur(gray_f ** 2, (15, 15), 0)
    local_var = blur_sq - blur ** 2
    local_var = np.clip(local_var, 0, None)

    # Smooth regions have low variance — road is smoother than textured walls
    smooth_mask = local_var < 400

    # --- Step 3: Trapezoidal spatial prior ---
    # Road in dashcam: wide at bottom, narrows toward vanishing point ~center-top
    spatial = np.zeros((h, w), dtype=np.uint8)
    vanish_y = int(h * 0.40)  # approximate vanishing point height
    vanish_x = int(w * 0.50)
    bot_y = int(h * 0.92)     # above dashboard

    # Trapezoid: narrow at top, wide at bottom
    trap_top_half = int(w * 0.12)   # narrow opening at vanishing point
    trap_bot_half = int(w * 0.50)   # full width at bottom

    pts = np.array([
        [vanish_x - trap_top_half, vanish_y],
        [vanish_x + trap_top_half, vanish_y],
        [vanish_x + trap_bot_half, bot_y],
        [vanish_x - trap_bot_half, bot_y],
    ], dtype=np.int32)
    cv2.fillConvexPoly(spatial, pts, 255)

    # --- Step 4: Combine all three cues ---
    combined = (color_mask.astype(np.uint8) * 255) & (smooth_mask.astype(np.uint8) * 255) & spatial

    # --- Step 5: Morphological cleanup ---
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel_close)
    combined = cv2.morphologyEx(combined, cv2.MORPH_OPEN, kernel_open)

    # --- Step 6: Keep largest component touching bottom-center ---
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(combined, connectivity=8)

    if num_labels <= 1:
        return combined

    # Seed from bottom-center strip
    seed_top = int(h * 0.78)
    seed_left = int(w * 0.30)
    seed_right = int(w * 0.70)
    seed_region = labels[seed_top:bot_y, seed_left:seed_right]
    seed_labels = set(np.unique(seed_region)) - {0}

    if not seed_labels:
        # Fallback: largest component
        areas = stats[1:, cv2.CC_STAT_AREA]
        best = np.argmax(areas) + 1
        seed_labels = {best}

    final_mask = np.zeros_like(combined)
    for lbl in seed_labels:
        final_mask[labels == lbl] = 255

    # Final smoothing
    kernel_smooth = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_CLOSE, kernel_smooth)

    return final_mask


def create_preview(img_bgr, mask, alpha=0.4):
    """Overlay road mask on the original image (green tint)."""
    overlay = img_bgr.copy()
    overlay[mask > 0] = (overlay[mask > 0] * (1 - alpha) +
                          np.array([0, 200, 0]) * alpha).astype(np.uint8)

    # Draw mask contour
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)

    return overlay


def main():
    parser = argparse.ArgumentParser(description="Auto-annotate roads in dashcam images.")
    parser.add_argument("--input", type=str, default="/content/drive/MyDrive/ICS - Sem2/CV/sampled_ghana_images",
                        help="Input image directory")
    parser.add_argument("--output", type=str, default="/content/drive/MyDrive/ICS - Sem2/CV/sampled_ghana_test_labels",
                        help="Output mask directory")
    parser.add_argument("--preview", action="store_true",
                        help="Save overlay previews alongside masks")
    parser.add_argument("--preview_dir", type=str, default="sample_test_preview",
                        help="Directory for preview images")

    # In a notebook environment, sys.argv can contain extra arguments not intended for argparse.
    # We explicitly pass an empty list to parse_args to avoid this.
    if "ipykernel" in sys.modules:
        args = parser.parse_args([]) # Pass an empty list directly
    else:
        args = parser.parse_args() # Use default sys.argv for standalone execution

    # Collect images
    paths = sorted(
        glob.glob(os.path.join(args.input, "*.jpg")) +
        glob.glob(os.path.join(args.input, "*.jpeg")) +
        glob.glob(os.path.join(args.input, "*.png"))
    )
    if not paths:
        print(f"No images found in {args.input}/")
        return

    os.makedirs(args.output, exist_ok=True)
    if args.preview:
        os.makedirs(args.preview_dir, exist_ok=True)

    print(f"Auto-annotating {len(paths)} images ...\n")

    for path in paths:
        fname = os.path.basename(path)
        name, _ = os.path.splitext(fname)

        img_bgr = cv2.imread(path)
        if img_bgr is None:
            print(f"  Skipping {fname} (cannot read)")
            continue

        mask = annotate_road(img_bgr)

        # Road pixel percentage
        h, w = mask.shape
        road_pct = (mask > 0).sum() / (h * w) * 100

        # Save mask as PNG (black/white)
        mask_path = os.path.join(args.output, f"{name}.png")
        cv2.imwrite(mask_path, mask)

        print(f"  {fname}: road = {road_pct:.1f}% of image")

        # Save preview
        if args.preview:
            preview = create_preview(img_bgr, mask)
            preview_path = os.path.join(args.preview_dir, f"{name}_preview.jpg")
            cv2.imwrite(preview_path, preview)

    print(f"\nMasks saved to: {args.output}/")
    if args.preview:
        print(f"Previews saved to: {args.preview_dir}/")
    print("Done.")


if __name__ == "__main__":
    main()


### Test Deeplabv3 on local video

In [ ]:
import os
import glob

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
from dotenv import load_dotenv
import kagglehub
import sys
import subprocess
import gdown
CHECKPOINT = "best_deeplabv3plus_resnet101_cityscapes_os16.pth"
GDRIVE_ID = "1t7TC8mxQaFECt4jutdq_NMnWxdm6B-Nb"

# Download Cityscapes checkpoint if needed
if not os.path.isfile(CHECKPOINT):
    print("Downloading Cityscapes-pretrained checkpoint …")
    gdown.download(id=GDRIVE_ID, output=CHECKPOINT, quiet=False)

In [ ]:
"""
Evaluate the BASE (pretrained Cityscapes 19-class) DeepLabV3+ model on local test images.

Uses class 0 = road from the original 19-class Cityscapes output.
No fine-tuning applied — this is the baseline for comparison.

Usage:
    python eval_segment_base.py --images sample_test/ --labels sample_test_labels/
    python eval_segment_base.py --ckpt best_deeplabv3plus_resnet101_cityscapes_os16.pth
"""

import os
import sys
import glob
import subprocess
import argparse

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
import gdown


# ── Args ──────────────────────────────────────────────────────────────────────

parser = argparse.ArgumentParser(description="Evaluate BASE 19-class segmentation model.")
parser.add_argument("--images", type=str, default="/content/drive/MyDrive/ICS - Sem2/CV/sampled_ghana_images",
                    help="Directory of test images")
parser.add_argument("--labels", type=str, default="/content/drive/MyDrive/ICS - Sem2/CV/sampled_ghana_test_labels",
                    help="Directory of ground-truth masks (binary PNGs from auto_annotate.py)")
parser.add_argument("--ckpt", type=str,
                    default="best_deeplabv3plus_resnet101_cityscapes_os16.pth",
                    help="Base Cityscapes checkpoint (19-class)")
parser.add_argument("--out", type=str, default="/content/drive/MyDrive/ICS - Sem2/CV/local_base_model_eval",
                    help="Output directory for visual results")
parser.add_argument("--img_h", type=int, default=512)
parser.add_argument("--img_w", type=int, default=1024)

# Colab-friendly: ignore notebook kernel args
if "ipykernel" in sys.modules:
    args = parser.parse_args([])
else:
    args = parser.parse_args()

# CHECKPOINT = "best_deeplabv3plus_resnet101_cityscapes_os16.pth"
# GDRIVE_ID = "1t7TC8mxQaFECt4jutdq_NMnWxdm6B-Nb"

# # Download Cityscapes checkpoint if needed
# if not os.path.isfile(CHECKPOINT):
#     print("Downloading Cityscapes-pretrained checkpoint …")
#     gdown.download(id=GDRIVE_ID, output=CHECKPOINT, quiet=False)

# ── Load model (BASE 19-class — no head replacement) ─────────────────────────

REPO_DIR = "DeepLabV3Plus-Pytorch"
if not os.path.isdir(REPO_DIR):
    print("Cloning DeepLabV3Plus-Pytorch ...")
    subprocess.run(
        ["git", "clone", "https://github.com/VainF/DeepLabV3Plus-Pytorch.git"],
        check=True,
    )

sys.path.insert(0, REPO_DIR)
from network.modeling import deeplabv3plus_resnet101

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

print(f"Loading BASE checkpoint: {args.ckpt}")
model = deeplabv3plus_resnet101(num_classes=19, output_stride=16)

ckpt = torch.load(args.ckpt, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.to(device).eval()
print(f"  Loaded 19-class Cityscapes model (road = class 0)\n")

# Cityscapes class 0 = road
ROAD_CLASS = 0


# ── Transforms ────────────────────────────────────────────────────────────────

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

img_transform = transforms.Compose([
    transforms.Resize((args.img_h, args.img_w)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# ── Collect image/label pairs ─────────────────────────────────────────────────

image_paths = sorted(
    glob.glob(os.path.join(args.images, "*.jpg")) +
    glob.glob(os.path.join(args.images, "*.jpeg")) +
    glob.glob(os.path.join(args.images, "*.png"))
)

if not image_paths:
    print(f"No images found in {args.images}/")
    sys.exit(1)

pairs = []
for img_path in image_paths:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    label_path = os.path.join(args.labels, f"{stem}.png")
    if os.path.isfile(label_path):
        pairs.append((img_path, label_path))
    else:
        print(f"  Warning: no label for {os.path.basename(img_path)}, skipping")

print(f"Found {len(pairs)} image/label pairs\n")
if not pairs:
    sys.exit(1)

os.makedirs(args.out, exist_ok=True)


# ── Metrics helpers ───────────────────────────────────────────────────────────

def compute_binary_metrics(pred_mask, gt_mask):
    """Compute IoU, pixel accuracy, precision, recall, F1 for binary road masks."""
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    union = tp + fp + fn
    iou = tp / union if union > 0 else (1.0 if fn == 0 else 0.0)

    total = tp + fp + fn + tn
    accuracy = (tp + tn) / total if total > 0 else 0.0

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "iou": float(iou),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }


def compute_ap(prob_map, gt_mask, num_thresholds=200):
    """Compute Average Precision from road probability map and binary GT."""
    thresholds = np.linspace(1.0, 0.0, num_thresholds)
    precisions = []
    recalls = []

    gt_flat = gt_mask.flatten().astype(bool)
    prob_flat = prob_map.flatten()

    for t in thresholds:
        pred = prob_flat >= t
        tp = np.logical_and(pred, gt_flat).sum()
        fp = np.logical_and(pred, ~gt_flat).sum()
        fn = np.logical_and(~pred, gt_flat).sum()

        prec = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        precisions.append(prec)
        recalls.append(rec)

    precisions = np.array(precisions)
    recalls = np.array(recalls)

    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])

    ap = 0.0
    for i in range(1, len(recalls)):
        dr = recalls[i] - recalls[i - 1]
        if dr > 0:
            ap += precisions[i] * dr

    return float(ap), precisions, recalls


# ── Visualization ─────────────────────────────────────────────────────────────

def create_comparison(img_bgr, pred_mask, gt_mask):
    """
    Create a 3-panel comparison: original | GT overlay | prediction overlay.
    Green = true positive, Red = false positive, Blue = false negative.
    """
    gt_vis = img_bgr.copy()
    gt_vis[gt_mask] = (gt_vis[gt_mask] * 0.6 + np.array([0, 180, 0]) * 0.4).astype(np.uint8)

    pred_vis = img_bgr.copy()
    tp = pred_mask & gt_mask
    fp = pred_mask & ~gt_mask
    fn = ~pred_mask & gt_mask

    pred_vis[tp] = (pred_vis[tp] * 0.5 + np.array([0, 200, 0]) * 0.5).astype(np.uint8)
    pred_vis[fp] = (pred_vis[fp] * 0.5 + np.array([0, 0, 220]) * 0.5).astype(np.uint8)
    pred_vis[fn] = (pred_vis[fn] * 0.5 + np.array([220, 0, 0]) * 0.5).astype(np.uint8)

    font = cv2.FONT_HERSHEY_SIMPLEX
    for panel, label in [(img_bgr, "Original"), (gt_vis, "Ground Truth"),
                         (pred_vis, "Prediction (Base)")]:
        cv2.putText(panel, label, (10, 30), font, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(panel, label, (10, 30), font, 0.8, (0, 0, 0), 1, cv2.LINE_AA)

    return np.hstack([img_bgr, gt_vis, pred_vis])


# ── Run inference ─────────────────────────────────────────────────────────────

print(f"Running BASE model inference on {len(pairs)} images ...\n")

all_metrics = []
all_aps = []

for img_path, label_path in pairs:
    fname = os.path.basename(img_path)
    stem = os.path.splitext(fname)[0]

    img_pil = Image.open(img_path).convert("RGB")

    # Load GT mask (binary: 255=road, 0=not-road)
    gt_raw = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)
    gt_resized = cv2.resize(gt_raw, (args.img_w, args.img_h), interpolation=cv2.INTER_NEAREST)
    gt_mask = gt_resized > 127

    # Run model
    inp = img_transform(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inp)                          # (1, 19, H, W)
        probs = F.softmax(logits, dim=1)             # (1, 19, H, W)
        road_prob = probs[0, ROAD_CLASS].cpu().numpy()  # class 0 = road
        pred_mask = logits.argmax(dim=1).squeeze(0).cpu().numpy() == ROAD_CLASS

    # Compute metrics
    metrics = compute_binary_metrics(pred_mask, gt_mask)
    ap, _, _ = compute_ap(road_prob, gt_mask)
    metrics["ap"] = ap
    all_metrics.append(metrics)
    all_aps.append(ap)

    print(f"  {fname}:")
    print(f"    IoU={metrics['iou']:.4f}  Acc={metrics['accuracy']:.4f}  "
          f"Prec={metrics['precision']:.4f}  Rec={metrics['recall']:.4f}  "
          f"F1={metrics['f1']:.4f}  AP={ap:.4f}")

    # Save visual comparison
    img_bgr = cv2.imread(img_path)
    img_viz = cv2.resize(img_bgr, (args.img_w, args.img_h))

    comparison = create_comparison(img_viz, pred_mask, gt_mask)
    cv2.imwrite(os.path.join(args.out, f"{stem}_eval_base.jpg"), comparison)


# ── Summary ───────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("EVALUATION SUMMARY — BASE MODEL (19-class Cityscapes)")
print("=" * 60)

mean_iou = np.mean([m["iou"] for m in all_metrics])
mean_acc = np.mean([m["accuracy"] for m in all_metrics])
mean_prec = np.mean([m["precision"] for m in all_metrics])
mean_rec = np.mean([m["recall"] for m in all_metrics])
mean_f1 = np.mean([m["f1"] for m in all_metrics])
mean_ap = np.mean(all_aps)

print(f"  Images evaluated:  {len(pairs)}")
print(f"  Mean IoU (road):   {mean_iou:.4f}")
print(f"  Mean Pixel Acc:    {mean_acc:.4f}")
print(f"  Mean Precision:    {mean_prec:.4f}")
print(f"  Mean Recall:       {mean_rec:.4f}")
print(f"  Mean F1:           {mean_f1:.4f}")
print(f"  mAP (road class):  {mean_ap:.4f}")

print(f"\n  Per-image IoU:")
for (img_path, _), m in zip(pairs, all_metrics):
    print(f"    {os.path.basename(img_path)}: IoU={m['iou']:.4f}  AP={m['ap']:.4f}")

print(f"\nVisual results saved to: {args.out}/")
print("  Green = true positive, Red = false positive, Blue = false negative")
print("=" * 60)


### Train deeplabv3+ on original cityscape data

In [ ]:
import os
import glob
import random

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from tqdm import tqdm
from dotenv import load_dotenv
import kagglehub
import sys
import subprocess
import gdown

# ── 1. Download dataset ─────────────────────────────────────────────────────


load_dotenv()
from google.colab import userdata
api_key = userdata.get('KAGGLE_API_KEY')
# api_key = os.environ.get("KAGGLE_API_KEY") or os.environ.get("KAGGLE_KEY")
if api_key:
    os.environ["KAGGLE_KEY"] = api_key
    if not os.environ.get("KAGGLE_USERNAME"):
        os.environ["KAGGLE_USERNAME"] = "__token__"

print("Downloading Cityscapes dataset …")
dataset_path = kagglehub.dataset_download("shuvoalok/cityscapes")
print(f"Dataset path: {dataset_path}")

# Locate train and val directories
train_img_dir = os.path.join(dataset_path, "train", "img")
train_seg_dir = os.path.join(dataset_path, "train", "label")
val_img_dir = os.path.join(dataset_path, "val", "img")
val_seg_dir = os.path.join(dataset_path, "val", "label")

# Fallback: search recursively if the expected structure differs
for split, dirs in [("train", [train_img_dir, train_seg_dir]),
                    ("val", [val_img_dir, val_seg_dir])]:
    if not os.path.isdir(dirs[0]):
        for root, subdirs, _ in os.walk(dataset_path):
            if split in subdirs:
                candidate = os.path.join(root, split)
                sub = os.listdir(candidate)
                if "img" in sub:
                    dirs[0] = os.path.join(candidate, "img")
                    dirs[1] = os.path.join(candidate, "label")
                    if split == "train":
                        train_img_dir, train_seg_dir = dirs
                    else:
                        val_img_dir, val_seg_dir = dirs
                    break
        print(f"Resolved {split} dirs → images: {dirs[0]}, labels: {dirs[1]}")

assert os.path.isdir(train_img_dir), f"Train image dir not found: {train_img_dir}"
assert os.path.isdir(train_seg_dir), f"Train label dir not found: {train_seg_dir}"
assert os.path.isdir(val_img_dir), f"Val image dir not found: {val_img_dir}"
assert os.path.isdir(val_seg_dir), f"Val label dir not found: {val_seg_dir}"

# ── 2. Dataset class ────────────────────────────────────────────────────────

ROAD_COLOR = (128, 64, 128)
COLOR_TOLERANCE = 5
IMG_SIZE = (512, 1024)  # H, W
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

img_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

img_transform_aug = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class CityscapesRoad(Dataset):
    def __init__(self, img_dir, seg_dir, augment=False):
        self.images = sorted(glob.glob(os.path.join(img_dir, "*.*")))
        self.labels = sorted(glob.glob(os.path.join(seg_dir, "*.*")))
        assert len(self.images) == len(self.labels), (
            f"Mismatch: {len(self.images)} images vs {len(self.labels)} labels"
        )
        self.augment = augment
        self.transform = img_transform_aug if augment else img_transform
        print(f"Found {len(self.images)} image/label pairs (augment={augment})")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        seg = Image.open(self.labels[idx]).convert("RGB")

        # Random horizontal flip (applied consistently to both)
        if self.augment and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            seg = seg.transpose(Image.FLIP_LEFT_RIGHT)

        img = self.transform(img)
        seg = seg.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.NEAREST)

        # Binary road mask from RGB color (tolerance for compression artifacts)
        seg_np = np.array(seg).astype(np.int16)
        diff = np.abs(seg_np - np.array(ROAD_COLOR, dtype=np.int16))
        road_mask = (diff.max(axis=2) <= COLOR_TOLERANCE).astype(np.int64)

        return img, torch.from_numpy(road_mask)


# Create datasets
train_dataset = CityscapesRoad(train_img_dir, train_seg_dir, augment=True)

# Use a small random subset of val for monitoring training progress
val_full = CityscapesRoad(val_img_dir, val_seg_dir, augment=False)
VAL_SUBSET_SIZE = min(50, len(val_full))
val_indices = random.sample(range(len(val_full)), VAL_SUBSET_SIZE)
val_dataset = Subset(val_full, val_indices)
print(f"Using {VAL_SUBSET_SIZE} val images for training monitoring")

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ── 3. Model ────────────────────────────────────────────────────────────────

REPO_DIR = "DeepLabV3Plus-Pytorch"
CHECKPOINT = "best_deeplabv3plus_resnet101_cityscapes_os16.pth"
GDRIVE_ID = "1t7TC8mxQaFECt4jutdq_NMnWxdm6B-Nb"

if not os.path.isdir(REPO_DIR):
    print("Cloning DeepLabV3Plus-Pytorch …")
    subprocess.run(["git", "clone", "https://github.com/VainF/DeepLabV3Plus-Pytorch.git"], check=True)

if not os.path.isfile(CHECKPOINT):
    print("Downloading Cityscapes-pretrained checkpoint …")
    gdown.download(id=GDRIVE_ID, output=CHECKPOINT, quiet=False)

sys.path.insert(0, REPO_DIR)
from network.modeling import deeplabv3plus_resnet101

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pretrained 19-class model
model = deeplabv3plus_resnet101(num_classes=19, output_stride=16)
checkpoint_data = torch.load(CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(checkpoint_data["model_state"])

# Replace classifier head: 19 → 2 classes (road / not-road)
NUM_CLASSES = 2
model.classifier.classifier[3] = nn.Conv2d(256, NUM_CLASSES, kernel_size=1)
nn.init.kaiming_normal_(model.classifier.classifier[3].weight)
nn.init.zeros_(model.classifier.classifier[3].bias)

# Freeze backbone, only train classifier head
for param in model.backbone.parameters():
    param.requires_grad = False

model = model.to(device)

# ── 4. Training setup ───────────────────────────────────────────────────────

NUM_EPOCHS = 10
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.5
)

best_iou = 0.0
SAVE_PATH = "/content/drive/MyDrive/ICS - Sem2/CV/deeplab_cityscape_best_model.pth"


def compute_iou(preds, targets):
    """Compute mean IoU for road class across a batch."""
    ious = []
    for p, g in zip(preds, targets):
        p = p.astype(bool)
        g = g.astype(bool)
        intersection = np.logical_and(p, g).sum()
        union = np.logical_or(p, g).sum()
        if union == 0:
            ious.append(1.0 if intersection == 0 else 0.0)
        else:
            ious.append(intersection / union)
    return ious


# ── 5. Training loop ────────────────────────────────────────────────────────

print(f"\nStarting training for {NUM_EPOCHS} epochs …\n")

for epoch in range(NUM_EPOCHS):
    # -- Train --
    model.train()
    # Keep backbone frozen in eval mode (for BatchNorm)
    model.backbone.eval()

    train_loss = 0.0
    train_steps = 0

    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]"):
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        output = model(imgs)  # (B, 2, H, W)
        loss = criterion(output, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_steps += 1

    avg_train_loss = train_loss / train_steps

    # -- Validate --
    model.eval()
    val_ious = []

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [val]"):
            imgs = imgs.to(device)
            output = model(imgs)  # (B, 2, H, W)
            preds = (output.argmax(dim=1) == 1).cpu().numpy()  # class 1 = road
            val_ious.extend(compute_iou(preds, masks.numpy()))

    val_iou = float(np.mean(val_ious))
    scheduler.step(val_iou)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: train_loss={avg_train_loss:.4f}, "
          f"val_IoU={val_iou:.4f}, lr={current_lr:.6f}")

    # Save best model
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save({
            "epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_iou": val_iou,
        }, SAVE_PATH)
        print(f"  → Saved new best model (IoU={val_iou:.4f})")

# ── 6. Summary ──────────────────────────────────────────────────────────────

print("\n" + "=" * 50)
print("TRAINING COMPLETE")
print("=" * 50)
print(f"  Best val IoU: {best_iou:.4f}")
print(f"  Checkpoint:   {SAVE_PATH}")
print("=" * 50)


### Augment CityScape Dataset

In [ ]:
import os
import glob
import random
import argparse
import shutil
import sys # Import sys

import cv2
import numpy as np
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
from dotenv import load_dotenv
import kagglehub

# ── Constants ────────────────────────────────────────────────────────────────

ROAD_COLOR = (128, 64, 128)
COLOR_TOLERANCE = 5
IMG_SIZE = (512, 1024)  # H, W


# ── GhanaAugmentor ──────────────────────────────────────────────────────────

class GhanaAugmentor:
    """Domain-specific augmentations to bridge the Cityscapes → Ghana gap."""

    def warm_color_shift(self, img_np):
        """Shift color temperature warmer: boost red, reduce blue."""
        img = img_np.astype(np.float32)
        red_boost = random.uniform(5, 20)
        blue_reduce = random.uniform(5, 20)
        img[:, :, 0] = np.clip(img[:, :, 0] + red_boost, 0, 255)   # R
        img[:, :, 2] = np.clip(img[:, :, 2] - blue_reduce, 0, 255)  # B
        return img.astype(np.uint8)

    def saturation_boost(self, img_np):
        """Boost saturation to simulate vivid tropical vegetation."""
        hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV).astype(np.float32)
        factor = random.uniform(1.2, 1.6)
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * factor, 0, 255)
        hsv = hsv.astype(np.uint8)
        return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

    def road_marking_degradation(self, img_np, road_mask):
        """Blur road pixels to fade lane markings."""
        ksize = random.choice(range(5, 16, 2))  # odd kernel 5–15
        blurred = cv2.GaussianBlur(img_np, (ksize, ksize), 0)
        mask_3c = road_mask[:, :, None].astype(bool)
        img_np = np.where(mask_3c, blurred, img_np)
        return img_np

    def simulated_potholes(self, img_np, road_mask):
        """Stamp dark ellipses on road regions to simulate potholes/patches."""
        num = random.randint(3, 8)
        road_ys, road_xs = np.where(road_mask > 0)
        if len(road_ys) == 0:
            return img_np
        img_out = img_np.copy()
        for _ in range(num):
            idx = random.randint(0, len(road_ys) - 1)
            cy, cx = int(road_ys[idx]), int(road_xs[idx])
            axes = (random.randint(10, 40), random.randint(10, 40))
            angle = random.randint(0, 180)
            intensity = random.randint(30, 80)
            color = (intensity, intensity, intensity)
            cv2.ellipse(img_out, (cx, cy), axes, angle, 0, 360, color, -1)
        # Only keep changes inside road mask
        mask_3c = road_mask[:, :, None].astype(bool)
        img_np = np.where(mask_3c, img_out, img_np)
        return img_np

    def edge_clutter(self, img_np, road_mask):
        """Add noise along road mask boundaries to simulate edge clutter."""
        band_width = random.randint(20, 40)
        kernel = np.ones((band_width, band_width), np.uint8)
        dilated = cv2.dilate(road_mask.astype(np.uint8), kernel, iterations=1)
        eroded = cv2.erode(road_mask.astype(np.uint8), kernel, iterations=1)
        edge_band = ((dilated - eroded) > 0)
        sigma = random.uniform(20, 40)
        noise = np.random.normal(0, sigma, img_np.shape).astype(np.float32)
        img_float = img_np.astype(np.float32)
        edge_3c = edge_band[:, :, None]
        img_float = np.where(edge_3c, img_float + noise, img_float)
        return np.clip(img_float, 0, 255).astype(np.uint8)

    def __call__(self, img_np, road_mask):
        """Apply Ghana-specific augmentations randomly.

        Global color transforms (1-3): 50% probability each.
        Road-aware transforms (4-6): 40% probability each.
        """
        # 1. Brightness/contrast via ColorJitter
        if random.random() < 0.5:
            pil_img = Image.fromarray(img_np)
            jitter = transforms.ColorJitter(brightness=0.4, contrast=0.4)
            pil_img = jitter(pil_img)
            img_np = np.array(pil_img)

        # 2. Warm color temperature shift
        if random.random() < 0.5:
            img_np = self.warm_color_shift(img_np)

        # 3. Saturation boost
        if random.random() < 0.5:
            img_np = self.saturation_boost(img_np)

        # 4. Road marking degradation
        if random.random() < 0.4:
            img_np = self.road_marking_degradation(img_np, road_mask)

        # 5. Simulated potholes/patches
        if random.random() < 0.4:
            img_np = self.simulated_potholes(img_np, road_mask)

        # 6. Edge clutter
        if random.random() < 0.4:
            img_np = self.edge_clutter(img_np, road_mask)

        return img_np


# ── Helpers ──────────────────────────────────────────────────────────────────

def extract_road_mask(seg_np):
    """Binary road mask from an RGB segmentation array."""
    seg16 = seg_np.astype(np.int16)
    diff = np.abs(seg16 - np.array(ROAD_COLOR, dtype=np.int16))
    return (diff.max(axis=2) <= COLOR_TOLERANCE).astype(np.uint8)


def resolve_dirs(dataset_path):
    """Locate train/val img/label dirs with fallback search."""
    train_img_dir = os.path.join(dataset_path, "train", "img")
    train_seg_dir = os.path.join(dataset_path, "train", "label")
    val_img_dir = os.path.join(dataset_path, "val", "img")
    val_seg_dir = os.path.join(dataset_path, "val", "label")

    for split, dirs in [("train", [train_img_dir, train_seg_dir]),
                        ("val", [val_img_dir, val_seg_dir])]:
        if not os.path.isdir(dirs[0]):
            for root, subdirs, _ in os.walk(dataset_path):
                if split in subdirs:
                    candidate = os.path.join(root, split)
                    sub = os.listdir(candidate)
                    if "img" in sub:
                        dirs[0] = os.path.join(candidate, "img")
                        dirs[1] = os.path.join(candidate, "label")
                        if split == "train":
                            train_img_dir, train_seg_dir = dirs
                        else:
                            val_img_dir, val_seg_dir = dirs
                        break
            print(f"Resolved {split} dirs → images: {dirs[0]}, labels: {dirs[1]}")

    return train_img_dir, train_seg_dir, val_img_dir, val_seg_dir


# ── Main ─────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="Augment Cityscapes images with Ghana-specific transforms and save to disk."
    )
    parser.add_argument("--out", type=str, default="/content/drive/MyDrive/ICS - Sem2/CV/cityscape",
                        help="Output directory (default: ghana_augmented)")
    parser.add_argument("--comparisons", type=int, default=5,
                        help="Number of side-by-side comparison images to save (default: 5)")

    # In a notebook environment, sys.argv can contain extra arguments not intended for argparse.
    # We explicitly pass an empty list to parse_args to avoid this.
    args = parser.parse_args(args=[] if "ipykernel" in sys.modules else None)

    # ── Download / locate dataset ────────────────────────────────────────
    load_dotenv()
    from google.colab import userdata
    api_key = userdata.get('KAGGLE_API_KEY')
    if api_key:
        os.environ["KAGGLE_KEY"] = api_key
        if not os.environ.get("KAGGLE_USERNAME"):
            os.environ["KAGGLE_USERNAME"] = "__token__"

    print("Downloading Cityscapes dataset …")
    dataset_path = kagglehub.dataset_download("shuvoalok/cityscapes")
    print(f"Dataset path: {dataset_path}")

    train_img_dir, train_seg_dir, val_img_dir, val_seg_dir = resolve_dirs(dataset_path)

    for d in [train_img_dir, train_seg_dir, val_img_dir, val_seg_dir]:
        assert os.path.isdir(d), f"Directory not found: {d}"

    # ── Create output directories ────────────────────────────────────────
    out_train_img = os.path.join(args.out, "train", "img")
    out_train_lbl = os.path.join(args.out, "train", "label")
    out_val_img = os.path.join(args.out, "val", "img")
    out_val_lbl = os.path.join(args.out, "val", "label")
    out_cmp = os.path.join(args.out, "comparisons")

    for d in [out_train_img, out_train_lbl, out_val_img, out_val_lbl, out_cmp]:
        os.makedirs(d, exist_ok=True)

    augmentor = GhanaAugmentor()

    # ── Augment training images ──────────────────────────────────────────
    train_images = sorted(glob.glob(os.path.join(train_img_dir, "*.*")))
    train_labels = sorted(glob.glob(os.path.join(train_seg_dir, "*.*")))
    assert len(train_images) == len(train_labels), "Train image/label count mismatch"

    comparison_indices = set(random.sample(range(len(train_images)),
                                           min(args.comparisons, len(train_images))))

    print(f"\nAugmenting {len(train_images)} training images …")
    for i, (img_path, lbl_path) in enumerate(tqdm(zip(train_images, train_labels),
                                                   total=len(train_images),
                                                   desc="Augmenting train")):
        img = Image.open(img_path).convert("RGB")
        seg = Image.open(lbl_path).convert("RGB")

        # Random horizontal flip (applied consistently to both)
        if random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            seg = seg.transpose(Image.FLIP_LEFT_RIGHT)

        img = img.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.BILINEAR)
        seg = seg.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.NEAREST)

        img_np = np.array(img)
        seg_np = np.array(seg)
        road_mask = extract_road_mask(seg_np)

        original = img_np.copy()
        augmented = augmentor(img_np, road_mask)

        # Save augmented image and label
        fname = os.path.basename(img_path)
        name, _ = os.path.splitext(fname)
        Image.fromarray(augmented).save(os.path.join(out_train_img, f"{name}.png"))
        seg.save(os.path.join(out_train_lbl, f"{name}.png"))

        # Save comparison if selected
        if i in comparison_indices:
            combined = np.concatenate([original, augmented], axis=1)
            Image.fromarray(combined).save(os.path.join(out_cmp, f"compare_{name}.png"))

    # ── Copy val images (resize only, no augmentation) ───────────────────
    val_images = sorted(glob.glob(os.path.join(val_img_dir, "*.*")))
    val_labels = sorted(glob.glob(os.path.join(val_seg_dir, "*.*")))
    assert len(val_images) == len(val_labels), "Val image/label count mismatch"

    print(f"\nResizing {len(val_images)} val images (no augmentation) …")
    for img_path, lbl_path in tqdm(zip(val_images, val_labels),
                                    total=len(val_images), desc="Resizing val"):
        img = Image.open(img_path).convert("RGB")
        seg = Image.open(lbl_path).convert("RGB")

        img = img.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.BILINEAR)
        seg = seg.resize((IMG_SIZE[1], IMG_SIZE[0]), Image.NEAREST)

        fname = os.path.basename(img_path)
        name, _ = os.path.splitext(fname)
        img.save(os.path.join(out_val_img, f"{name}.png"))
        seg.save(os.path.join(out_val_lbl, f"{name}.png"))

    # ── Summary ──────────────────────────────────────────────────────────
    print(f"\nDone!")
    print(f"  Augmented train: {out_train_img} ({len(train_images)} images)")
    print(f"  Val (resized):   {out_val_img} ({len(val_images)} images)")
    print(f"  Comparisons:     {out_cmp} ({len(comparison_indices)} side-by-side images)")
    print(f"\nInspect comparisons, then run:  python train_ghana.py --data {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
import os
import glob
import random
import argparse

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from tqdm import tqdm
import sys
import subprocess
import gdown

# ── Arguments ────────────────────────────────────────────────────────────────

parser = argparse.ArgumentParser(description="Train DeepLabV3+ on augmented Cityscapes.")
parser.add_argument("--data", type=str, default="/content/drive/MyDrive/ICS - Sem2/CV/cityscape",
                    help="Path to augmented dataset (default: ghana_augmented)")
# In a notebook environment, sys.argv can contain extra arguments not intended for argparse.
# We explicitly pass an empty list to parse_args to avoid this.
args = parser.parse_args(args=[] if "ipykernel" in sys.modules else None)

# ── 1. Locate dataset ───────────────────────────────────────────────────────

train_img_dir = os.path.join(args.data, "train", "img")
train_seg_dir = os.path.join(args.data, "train", "label")
val_img_dir = os.path.join(args.data, "val", "img")
val_seg_dir = os.path.join(args.data, "val", "label")

for d in [train_img_dir, train_seg_dir, val_img_dir, val_seg_dir]:
    assert os.path.isdir(d), (
        f"Directory not found: {d}\n"
        f"Run augment_ghana.py first to generate the augmented dataset."
    )

# ── 2. Dataset class ────────────────────────────────────────────────────────

ROAD_COLOR = (128, 64, 128)
COLOR_TOLERANCE = 5
IMG_SIZE = (512, 1024)  # H, W
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class CityscapesRoadGhana(Dataset):
    """Loads pre-augmented, pre-resized images from disk."""

    def __init__(self, img_dir, seg_dir):
        self.images = sorted(glob.glob(os.path.join(img_dir, "*.*")))
        self.labels = sorted(glob.glob(os.path.join(seg_dir, "*.*")))
        assert len(self.images) == len(self.labels), (
            f"Mismatch: {len(self.images)} images vs {len(self.labels)} labels"
        )
        print(f"Found {len(self.images)} image/label pairs in {img_dir}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        seg = Image.open(self.labels[idx]).convert("RGB")

        img = img_transform(img)

        # Binary road mask from RGB color
        seg_np = np.array(seg).astype(np.int16)
        diff = np.abs(seg_np - np.array(ROAD_COLOR, dtype=np.int16))
        road_mask = (diff.max(axis=2) <= COLOR_TOLERANCE).astype(np.int64)

        return img, torch.from_numpy(road_mask)


# Create datasets
train_dataset = CityscapesRoadGhana(train_img_dir, train_seg_dir)

val_full = CityscapesRoadGhana(val_img_dir, val_seg_dir)
VAL_SUBSET_SIZE = min(50, len(val_full))
val_indices = random.sample(range(len(val_full)), VAL_SUBSET_SIZE)
val_dataset = Subset(val_full, val_indices)
print(f"Using {VAL_SUBSET_SIZE} val images for training monitoring")

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ── 3. Model ────────────────────────────────────────────────────────────────

REPO_DIR = "DeepLabV3Plus-Pytorch"
CHECKPOINT = "best_deeplabv3plus_resnet101_cityscapes_os16.pth"
GDRIVE_ID = "1t7TC8mxQaFECt4jutdq_NMnWxdm6B-Nb"

if not os.path.isdir(REPO_DIR):
    print("Cloning DeepLabV3Plus-Pytorch …")
    subprocess.run(["git", "clone", "https://github.com/VainF/DeepLabV3Plus-Pytorch.git"], check=True)

if not os.path.isfile(CHECKPOINT):
    print("Downloading Cityscapes-pretrained checkpoint …")
    gdown.download(id=GDRIVE_ID, output=CHECKPOINT, quiet=False)

sys.path.insert(0, REPO_DIR)
from network.modeling import deeplabv3plus_resnet101

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pretrained 19-class model
model = deeplabv3plus_resnet101(num_classes=19, output_stride=16)
checkpoint_data = torch.load(CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(checkpoint_data["model_state"])

# Replace classifier head: 19 → 2 classes (road / not-road)
NUM_CLASSES = 2
model.classifier.classifier[3] = nn.Conv2d(256, NUM_CLASSES, kernel_size=1)
nn.init.kaiming_normal_(model.classifier.classifier[3].weight)
nn.init.zeros_(model.classifier.classifier[3].bias)

# Freeze backbone, only train classifier head
for param in model.backbone.parameters():
    param.requires_grad = False

model = model.to(device)

# ── 4. Training loop ────────────────────────────────────────────────────────

NUM_EPOCHS = 10
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.5
)

best_iou = 0.0
SAVE_PATH = "/content/drive/MyDrive/ICS - Sem2/CV/augmented_city_scape_model.pth"


def compute_iou(preds, targets):
    """Compute mean IoU for road class across a batch."""
    ious = []
    for p, g in zip(preds, targets):
        p = p.astype(bool)
        g = g.astype(bool)
        intersection = np.logical_and(p, g).sum()
        union = np.logical_or(p, g).sum()
        if union == 0:
            ious.append(1.0 if intersection == 0 else 0.0)
        else:
            ious.append(intersection / union)
    return ious


print(f"\nStarting training for {NUM_EPOCHS} epochs …\n")

for epoch in range(NUM_EPOCHS):
    # -- Train --
    model.train()
    model.backbone.eval()

    train_loss = 0.0
    train_steps = 0

    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]"):
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        output = model(imgs)
        loss = criterion(output, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_steps += 1

    avg_train_loss = train_loss / train_steps

    # -- Validate --
    model.eval()
    val_ious = []

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [val]"):
            imgs = imgs.to(device)
            output = model(imgs)
            preds = (output.argmax(dim=1) == 1).cpu().numpy()
            val_ious.extend(compute_iou(preds, masks.numpy()))

    val_iou = float(np.mean(val_ious))
    scheduler.step(val_iou)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: train_loss={avg_train_loss:.4f}, "
          f"val_IoU={val_iou:.4f}, lr={current_lr:.6f}")

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save({
            "epoch": epoch + 1,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_iou": val_iou,
        }, SAVE_PATH)
        print(f"  → Saved new best model (IoU={val_iou:.4f})")

# ── 5. Summary ──────────────────────────────────────────────────────────────

print("\n" + "=" * 50)
print("TRAINING COMPLETE (Ghana-augmented)")
print("=" * 50)
print(f"  Best val IoU: {best_iou:.4f}")
print(f"  Checkpoint:   {SAVE_PATH}")
print("=" * 50)


### Test on local video

In [ ]:
import os
import sys
import glob
import argparse

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms


# ── Args ──────────────────────────────────────────────────────────────────────

parser = argparse.ArgumentParser(description="Evaluate road segmentation model.")
parser.add_argument("--images", type=str, default="/content/drive/MyDrive/ICS - Sem2/CV/sampled_ghana_images",
                    help="Directory of test images")
parser.add_argument("--labels", type=str, default="/content/drive/MyDrive/ICS - Sem2/CV/sampled_ghana_test_labels",
                    help="Directory of ground-truth masks (binary PNGs from auto_annotate.py)")
parser.add_argument("--ckpt", type=str, default="/content/drive/MyDrive/ICS - Sem2/CV/augmented_city_scape_model.pth",
                    help="Segmentation model checkpoint")
parser.add_argument("--out", type=str, default="/content/drive/MyDrive/ICS - Sem2/CV/local_augment_model_eval",
                    help="Output directory for visual results")
parser.add_argument("--img_h", type=int, default=512)
parser.add_argument("--img_w", type=int, default=1024)

# In a notebook environment, sys.argv can contain extra arguments not intended for argparse.
# We explicitly pass an empty list to parse_args to avoid this.
if "ipykernel" in sys.modules:
    args = parser.parse_args([]) # Pass an empty list directly
else:
    args = parser.parse_args() # Use default sys.argv for standalone execution

import subprocess

REPO_DIR = "DeepLabV3Plus-Pytorch"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/VainF/DeepLabV3Plus-Pytorch.git"], check=True)

# ── Load model ────────────────────────────────────────────────────────────────

REPO_DIR = "DeepLabV3Plus-Pytorch"
assert os.path.isdir(REPO_DIR), f"{REPO_DIR}/ not found. Clone it first."

sys.path.insert(0, REPO_DIR)
from network.modeling import deeplabv3plus_resnet101

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

print(f"Loading checkpoint: {args.ckpt}")
model = deeplabv3plus_resnet101(num_classes=19, output_stride=16)
model.classifier.classifier[3] = nn.Conv2d(256, 2, kernel_size=1)

ckpt = torch.load(args.ckpt, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.to(device).eval()

epoch = ckpt.get("epoch", "?")
val_iou = ckpt.get("val_iou", "?")
print(f"  Epoch {epoch}, training val_IoU={val_iou}\n")


# ── Transforms ────────────────────────────────────────────────────────────────

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

img_transform = transforms.Compose([
    transforms.Resize((args.img_h, args.img_w)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# ── Collect image/label pairs ─────────────────────────────────────────────────

image_paths = sorted(
    glob.glob(os.path.join(args.images, "*.jpg")) +
    glob.glob(os.path.join(args.images, "*.jpeg")) +
    glob.glob(os.path.join(args.images, "*.png"))
)

if not image_paths:
    print(f"No images found in {args.images}/")
    sys.exit(1)

# Match each image to its label by filename stem
pairs = []
for img_path in image_paths:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    # Labels are PNGs from auto_annotate.py
    label_path = os.path.join(args.labels, f"{stem}.png")
    if os.path.isfile(label_path):
        pairs.append((img_path, label_path))
    else:
        print(f"  Warning: no label for {os.path.basename(img_path)}, skipping")

print(f"Found {len(pairs)} image/label pairs\n")
if not pairs:
    sys.exit(1)

os.makedirs(args.out, exist_ok=True)


# ── Metrics helpers ───────────────────────────────────────────────────────────

def compute_binary_metrics(pred_mask, gt_mask):
    """
    Compute IoU, pixel accuracy, precision, recall, F1 for binary road masks.
    pred_mask, gt_mask: bool arrays of same shape.
    """
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    # IoU (road class)
    union = tp + fp + fn
    iou = tp / union if union > 0 else (1.0 if fn == 0 else 0.0)

    # Pixel accuracy
    total = tp + fp + fn + tn
    accuracy = (tp + tn) / total if total > 0 else 0.0

    # Precision, recall, F1
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "iou": float(iou),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
    }


def compute_ap(prob_map, gt_mask, num_thresholds=200):
    """
    Compute Average Precision from road probability map and binary GT.
    Sweeps confidence thresholds, computes precision-recall curve, then AP.
    """
    thresholds = np.linspace(1.0, 0.0, num_thresholds)
    precisions = []
    recalls = []

    gt_flat = gt_mask.flatten().astype(bool)
    prob_flat = prob_map.flatten()

    for t in thresholds:
        pred = prob_flat >= t
        tp = np.logical_and(pred, gt_flat).sum()
        fp = np.logical_and(pred, ~gt_flat).sum()
        fn = np.logical_and(~pred, gt_flat).sum()

        prec = tp / (tp + fp) if (tp + fp) > 0 else 1.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        precisions.append(prec)
        recalls.append(rec)

    precisions = np.array(precisions)
    recalls = np.array(recalls)

    # Monotonically decreasing precision (right to left)
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])

    # AP = area under PR curve
    ap = 0.0
    for i in range(1, len(recalls)):
        dr = recalls[i] - recalls[i - 1]
        if dr > 0:
            ap += precisions[i] * dr

    return float(ap), precisions, recalls


# ── Visualization ─────────────────────────────────────────────────────────────

def create_comparison(img_bgr, pred_mask, gt_mask):
    """
    Create a 3-panel comparison: original | GT overlay | prediction overlay.
    Green = true positive, Red = false positive, Blue = false negative.
    """
    h, w = img_bgr.shape[:2]

    # GT overlay (green)
    gt_vis = img_bgr.copy()
    gt_vis[gt_mask] = (gt_vis[gt_mask] * 0.6 + np.array([0, 180, 0]) * 0.4).astype(np.uint8)

    # Prediction overlay with error colors
    pred_vis = img_bgr.copy()
    tp = pred_mask & gt_mask
    fp = pred_mask & ~gt_mask
    fn = ~pred_mask & gt_mask

    pred_vis[tp] = (pred_vis[tp] * 0.5 + np.array([0, 200, 0]) * 0.5).astype(np.uint8)    # green = correct
    pred_vis[fp] = (pred_vis[fp] * 0.5 + np.array([0, 0, 220]) * 0.5).astype(np.uint8)    # red = false positive
    pred_vis[fn] = (pred_vis[fn] * 0.5 + np.array([220, 0, 0]) * 0.5).astype(np.uint8)    # blue = false negative

    # Labels
    font = cv2.FONT_HERSHEY_SIMPLEX
    for panel, label in [(img_bgr, "Original"), (gt_vis, "Ground Truth"), (pred_vis, "Prediction")]:
        cv2.putText(panel, label, (10, 30), font, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(panel, label, (10, 30), font, 0.8, (0, 0, 0), 1, cv2.LINE_AA)

    return np.hstack([img_bgr, gt_vis, pred_vis])


# ── Run inference ─────────────────────────────────────────────────────────────

print(f"Running inference on {len(pairs)} images ...\n")

all_metrics = []
all_aps = []

for img_path, label_path in pairs:
    fname = os.path.basename(img_path)
    stem = os.path.splitext(fname)[0]

    # Load image
    img_pil = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img_pil.size

    # Load GT mask (binary: 255=road, 0=not-road)
    gt_raw = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)
    # Resize GT to model input size for comparison
    gt_resized = cv2.resize(gt_raw, (args.img_w, args.img_h), interpolation=cv2.INTER_NEAREST)
    gt_mask = gt_resized > 127  # bool

    # Run model
    inp = img_transform(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inp)  # (1, 2, H, W)
        probs = F.softmax(logits, dim=1)  # (1, 2, H, W)
        road_prob = probs[0, 1].cpu().numpy()  # probability of road class
        pred_mask = logits.argmax(dim=1).squeeze(0).cpu().numpy() == 1  # bool

    # Compute metrics
    metrics = compute_binary_metrics(pred_mask, gt_mask)
    ap, _, _ = compute_ap(road_prob, gt_mask)
    metrics["ap"] = ap
    all_metrics.append(metrics)
    all_aps.append(ap)

    print(f"  {fname}:")
    print(f"    IoU={metrics['iou']:.4f}  Acc={metrics['accuracy']:.4f}  "
          f"Prec={metrics['precision']:.4f}  Rec={metrics['recall']:.4f}  "
          f"F1={metrics['f1']:.4f}  AP={ap:.4f}")

    # Save visual comparison
    # Resize pred and GT back to original image size for visualization
    img_bgr = cv2.imread(img_path)
    img_viz = cv2.resize(img_bgr, (args.img_w, args.img_h))
    pred_viz = pred_mask
    gt_viz = gt_mask

    comparison = create_comparison(img_viz, pred_viz, gt_viz)
    cv2.imwrite(os.path.join(args.out, f"{stem}_eval.jpg"), comparison)


# ── Summary ───────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)

mean_iou = np.mean([m["iou"] for m in all_metrics])
mean_acc = np.mean([m["accuracy"] for m in all_metrics])
mean_prec = np.mean([m["precision"] for m in all_metrics])
mean_rec = np.mean([m["recall"] for m in all_metrics])
mean_f1 = np.mean([m["f1"] for m in all_metrics])
mean_ap = np.mean(all_aps)

print(f"  Images evaluated:  {len(pairs)}")
print(f"  Mean IoU (road):   {mean_iou:.4f}")
print(f"  Mean Pixel Acc:    {mean_acc:.4f}")
print(f"  Mean Precision:    {mean_prec:.4f}")
print(f"  Mean Recall:       {mean_rec:.4f}")
print(f"  Mean F1:           {mean_f1:.4f}")
print(f"  mAP (road class):  {mean_ap:.4f}")

print(f"\n  Per-image IoU:")
for (img_path, _), m in zip(pairs, all_metrics):
    print(f"    {os.path.basename(img_path)}: IoU={m['iou']:.4f}  AP={m['ap']:.4f}")

print(f"\nVisual results saved to: {args.out}/")
print("  Green = true positive, Red = false positive, Blue = false negative")
print("=" * 60)


In [ ]:
import gdown
import zipfile
import os

# The Google Drive link for the zip file
link = "https://drive.google.com/file/d/1MbXwdiLvvGgiXn7J09g_gLK0lgEdMFHw/view?usp=sharing"

# Define the output path for the downloaded zip file
output_zip_file = "detect_ghana_test_images.zip"

# Define the directory where the contents will be extracted
extraction_dir = "/content/drive/MyDrive/ICS - Sem2/CV/detect_ghana_test_images"

print(f"Downloading {output_zip_file}...")
# Download the file using gdown, with fuzzy=True for Google Drive links
gdown.download(link, output_zip_file, quiet=False, fuzzy=True)

print(f"Extracting {output_zip_file} to {extraction_dir}...")
# Create the extraction directory if it doesn't exist
os.makedirs(extraction_dir, exist_ok=True)

# Extract the zip file
with zipfile.ZipFile(output_zip_file, 'r') as zip_ref:
    zip_ref.extractall(extraction_dir)

print("Download and extraction complete!")
print(f"Contents extracted to: {extraction_dir}/")


In [ ]:
import os
import random
import shutil

dir = '/content/drive/MyDrive/ICS - Sem2/CV/ghana_test_images' # Corrected path based on previous extraction
output_folder = '/content/drive/MyDrive/ICS - Sem2/CV/sampled_ghana_images' # New folder for sampled images
num_samples = 50

# Ensure the source directory exists
if not os.path.isdir(dir):
    print(f"Error: Source directory '{dir}' not found.")
else:
    # List all files in the directory
    all_images = [f for f in os.listdir(dir) if os.path.isfile(os.path.join(dir, f))]

    if not all_images:
        print(f"No images found in '{dir}'.")
    else:
        # Sample 'num_samples' random images, or all if less than num_samples
        sampled_images = random.sample(all_images, min(num_samples, len(all_images)))

        # Create the output folder if it doesn't exist
        os.makedirs(output_folder, exist_ok=True)

        # Copy sampled images to the new folder
        print(f"Sampling {len(sampled_images)} images and saving to '{output_folder}'...")
        for img_name in sampled_images:
            source_path = os.path.join(dir, img_name)
            destination_path = os.path.join(output_folder, img_name)
            shutil.copy(source_path, destination_path)
        print("Image sampling complete.")
